# Pipeline 2: Social Media Referrals (6 Prediction Targets)

**New Dawn Safehouse Management System — ML Pipeline**

---

## Table of Contents
1. Business Understanding & Problem Definition
2. Data Understanding & Exploration
3. Data Preparation & Feature Engineering
4. Modelling — 6 Regression Models (Ch. 6–9)
5. Hyperparameter Tuning (Ch. 11)
6. Model Evaluation & Feature Importance (Ch. 12–13)
7. Lookup Table Generation
8. Deployment — CSV Output & Web Integration (Ch. 15)

## 1. Business Understanding & Problem Definition

New Dawn's social media team publishes posts across Instagram, Facebook, TikTok, LinkedIn, Twitter, and WhatsApp. Each post has controllable features (platform, media type, CTA type, content topic, etc.) and measurable outcomes.

### Goal
Build **6 regression models** that predict post-level outcomes from controllable features:
1. `donation_referrals` — number of donations traced back to this post
2. `estimated_donation_value_php` — total PHP attributed to this post
3. `forwards` — shares/forwards
4. `profile_visits` — profile visits driven by this post
5. `engagement_rate` — likes + comments + shares / reach
6. `impressions` — total views

The predictions power the **Social Media Editor** page's real-time prediction cards, updating as the author changes post attributes.

### Success Criteria
- Positive R² on cross-validation for all 6 targets
- Reasonable RMSE relative to target standard deviation
- Lookup table ≤ 50,000 rows for fast API lookups

## 2. Data Understanding & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')

posts = pd.read_csv('../lighthouse_csv_v7/social_media_posts.csv')
print(f'Social media posts: {posts.shape}')
print(f'Columns: {posts.columns.tolist()}')
posts.head(3)

In [ ]:
# Platform & post type distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

posts['platform'].value_counts().plot.bar(ax=axes[0], color='#A2C9E1')
axes[0].set_title('Posts by Platform')
axes[0].tick_params(axis='x', rotation=45)

posts['post_type'].value_counts().plot.bar(ax=axes[1], color='#91B191')
axes[1].set_title('Posts by Type')
axes[1].tick_params(axis='x', rotation=45)

targets = ['donation_referrals', 'estimated_donation_value_php', 'impressions']
for t in targets:
    posts[t] = pd.to_numeric(posts[t], errors='coerce')
posts[targets].describe().T[['mean', 'std', 'min', 'max']].plot.barh(ax=axes[2])
axes[2].set_title('Target Variable Stats')

plt.tight_layout()
plt.show()

In [ ]:
# Target correlations
target_cols = ['donation_referrals', 'estimated_donation_value_php', 'forwards',
               'profile_visits', 'engagement_rate', 'impressions']
for c in target_cols:
    posts[c] = pd.to_numeric(posts[c], errors='coerce').fillna(0)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(posts[target_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Target Variable Correlations')
plt.tight_layout()
plt.show()

## 3. Data Preparation & Feature Engineering

We use the pipeline module's preparation functions:
- Boolean columns → 'Yes'/'No' strings for consistent encoding
- Categorical features → one-hot encoded
- Numeric features: `caption_length`, `num_hashtags`, `mentions_count`, `boost_budget_php`, `post_hour`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from pipelines.social_media_referrals import prepare_data, encode_features, TARGETS, CAT_COLS, NUM_COLS

df = prepare_data(posts)
X_encoded, feature_names = encode_features(df)

print(f'Prepared dataset: {df.shape}')
print(f'Encoded features: {X_encoded.shape}')
print(f'\nCategorical columns: {CAT_COLS}')
print(f'Numeric columns: {NUM_COLS}')
print(f'\nTarget variables: {TARGETS}')

## 4. Modelling — 6 Regression Models (Ch. 6–9)

For each of the 6 targets, we compare:
1. Ridge Regression (baseline)
2. Lasso Regression
3. Random Forest (Ch. 8)
4. Gradient Boosting (Ch. 9)
5. XGBoost
6. LightGBM

Using 5-Fold Cross-Validation with RMSE and R² metrics.

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from functions import evaluate_regressors

all_results = {}
best_models = {}

for target in TARGETS:
    print(f'\n{"=" * 50}')
    print(f'Target: {target}')
    print(f'{"=" * 50}')
    
    y = df[target].values
    
    models = {
        'Ridge': Ridge(alpha=1.0),
        'Lasso': Lasso(alpha=0.1, max_iter=5000),
        'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42),
        'XGBoost': XGBRegressor(n_estimators=200, random_state=42, verbosity=0),
        'LightGBM': LGBMRegressor(n_estimators=200, random_state=42, verbose=-1),
    }
    
    results = evaluate_regressors(X_encoded, y, models, cv=5)
    all_results[target] = results
    best_models[target] = results.iloc[0]['Model']
    print(results.to_string(index=False))
    print(f'Best: {best_models[target]}')

In [ ]:
# Summary of best model per target
summary = pd.DataFrame([
    {'Target': t, 'Best Model': best_models[t], 
     'RMSE': all_results[t].iloc[0].get('RMSE', ''),
     'R²': all_results[t].iloc[0].get('R2', '')}
    for t in TARGETS
])
print('\n── Best Model per Target ──')
summary

## 5. Hyperparameter Tuning (Ch. 11)

The pipeline module automatically tunes the best model per target using `GridSearchCV`. Here we show the tuning for the most important target: `donation_referrals`.

In [ ]:
from pipelines.social_media_referrals import train_target_model

# Tune donation_referrals model
y_dr = df['donation_referrals'].values
tuned_model_dr, _, best_name_dr = train_target_model(X_encoded, y_dr, 'donation_referrals')
print(f'\nTuned model: {best_name_dr}')
print(f'Params: {tuned_model_dr.get_params()}')

## 6. Model Evaluation & Feature Importance (Ch. 12–13)

In [ ]:
from functions import feature_importance_report

# Feature importances for donation_referrals model
if hasattr(tuned_model_dr, 'feature_importances_'):
    imp = feature_importance_report(tuned_model_dr, feature_names)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    imp.head(15).plot.barh(x='feature', y='importance', ax=ax, color='#A2C9E1', legend=False)
    ax.set_title(f'Top 15 Feature Importances — {best_name_dr} (donation_referrals)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('Model does not have feature_importances_ attribute (linear model)')

In [ ]:
# Actual vs Predicted scatter for donation_referrals
from sklearn.model_selection import cross_val_predict

y_pred_dr = cross_val_predict(tuned_model_dr, X_encoded, y_dr, cv=5)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_dr, y_pred_dr, alpha=0.4, color='#91B191', s=20)
ax.plot([0, y_dr.max()], [0, y_dr.max()], 'r--', lw=1)
ax.set_xlabel('Actual Donation Referrals')
ax.set_ylabel('Predicted Donation Referrals')
ax.set_title('Actual vs Predicted — Donation Referrals')
plt.tight_layout()
plt.show()

## 7. Lookup Table Generation

Instead of running the model in real-time, we pre-compute predictions for all realistic combinations of controllable features. This lookup table is stored as `social_post_predictions.csv`.

- Uses observed categorical combinations from historical data (no synthetic combos)
- Crosses with 25th/50th/75th percentile values for numeric features
- Capped at 50,000 rows for performance

In [ ]:
from pipelines.social_media_referrals import run

lookup_df, trained_models = run(posts)

print(f'\nLookup table shape: {lookup_df.shape}')
print(f'\nPredicted columns:')
pred_cols = [c for c in lookup_df.columns if c.startswith('predicted_')]
lookup_df[pred_cols].describe().round(2)

## 8. Deployment — CSV Output & Web Integration (Ch. 15)

The lookup CSV is consumed by:
- **Backend**: `CsvPredictionService` loads it into memory. `POST /api/predictions/ml/social-lookup` filters by platform, postType, mediaType, etc. and returns the closest matching predictions.
- **Frontend**: The Social Media Editor page sends debounced requests as the user changes post attributes, displaying predicted donation referrals, engagement rate, impressions, etc. in real-time prediction cards.

**Nightly refresh**: `run_all_pipelines.py` regenerates the lookup table at 2:00 AM.

In [ ]:
# Sample of the final output
display_cols = ['platform', 'post_type', 'media_type', 'content_topic'] + pred_cols
lookup_df[display_cols].head(10)

---

### Summary

| Target | Best Model | Key Finding |
|--------|-----------|-------------|
| donation_referrals | GBM | Platform and CTA type most important |
| estimated_donation_value_php | Ridge | Linear relationship with boost budget |
| forwards | LightGBM | Media type (Video/Reel) drives shares |
| profile_visits | GBM | Content topic matters most |
| engagement_rate | GBM | Sentiment tone influences engagement |
| impressions | Ridge | Boosted posts dominate impressions |

All 6 models are deployed as a single lookup CSV powering the Social Media Editor.